# External Validation on OAI — Multi-Model HKA Agreement

Runs the OAI full-limb external validation with **all six trained models** — the five cross-validation fold models and the final global model — and reports how much the result varies between independently trained models.

**Why.** Two training runs of the same recipe (same code, seed 42) produced global models whose sign-corrected OAI agreement differed by about 0.2 in ICC(2,1), because GPU training is not bit-deterministic. A single model's OAI score is therefore one draw from a distribution. This notebook measures that distribution instead of reporting one draw.

**What it reports**
1. SHA-256 of every weight file, checked against the recorded values — so each number below is tied to an exact set of weights.
2. Per-model agreement with the OAI clinical (OAISYS) HKA: MAE, RMSE, bias, 95% limits of agreement, ICC(2,1), Pearson r, Spearman ρ, gross-failure rate, and MAE by alignment stratum.
3. The same metrics on the **common knee set** (knees no model flagged), so models are compared on identical knees.
4. Aggregates across the five fold models (mean ± SD, range) and across all six, with the global model placed against the fold spread.
5. Between-model disagreement per knee.
6. A consistency check: the global model must reproduce the published single-model results (`hto_oai_external_validation.csv`).

**Setup.** The weights live in `weights/`, a sibling of `notebooks/`. The container only sees it if `docker-compose.yml` mounts it — add `- ./weights:/tf/weights` under `volumes:` and restart the container.

Preprocessing, landmark decoding and the laterality-consistent HKA sign are identical to `hto_correction_angles_oai_external_validation.ipynb` (commit `da2c1a1`).

## Imports & Configuration

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # required by deterministic cuBLAS; must precede CUDA init
import sys, math, random, hashlib, glob, datetime
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

for _ckd in ("CKD", "/tf/notebooks/CKD", os.path.join(os.getcwd(), "CKD")):
    if os.path.isdir(_ckd):
        sys.path.append(os.path.abspath(_ckd)); break
else:
    sys.path.append(os.path.abspath("CKD"))
from models import (
    Conformer_tiny_patch16_keypoint_half_heatmap,
    Conformer_small_patch16_keypoint_half_heatmap,
    Conformer_small_patch32_keypoint_half_heatmap,
    Conformer_base_patch16_keypoint_half_heatmap,
)
from utils import extract_coordinates

SEED          = 42
TARGET_SIZE   = 768
HEATMAP_SCALE = 0.5
MODEL_VARIANT = "small_p16"

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
# Inference only: request deterministic kernels, and warn (rather than fail) if an op has
# none, so any remaining source of run-to-run variation is named in the log.
torch.use_deterministic_algorithms(True, warn_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {device}"
      + (f" | {torch.cuda.get_device_name(0)}" if device.type == "cuda" else ""))

## Configuration — edit these

In [ ]:
OAI_IMAGE_DIR = "/tf/data/hka/oai_dicoms"                 # extracted <barcode>.dcm files
OAI_HKA_CSV   = "/tf/data/hka/oai_hka_groundtruth.csv"    # HKA ground truth

# weights/ is a sibling of notebooks/. Inside the container it is visible only if
# docker-compose.yml mounts it:   - ./weights:/tf/weights
WEIGHTS_DIR_CANDIDATES = ["/tf/weights", os.path.join("..", "weights"), "weights"]

MODELS = [                       # (label, file, role)
    ("fold1",  "best_model_fold1.pt",  "fold"),
    ("fold2",  "best_model_fold2.pt",  "fold"),
    ("fold3",  "best_model_fold3.pt",  "fold"),
    ("fold4",  "best_model_fold4.pt",  "fold"),
    ("fold5",  "best_model_fold5.pt",  "fold"),
    ("global", "best_model_global.pt", "global"),
]
ROLE = {label: role for label, _, role in MODELS}

# SHA-256 of each file as found in weights/ on 2026-09-19. The hashing cell checks every
# file against these; a mismatch means the weights on disk are not the ones recorded here.
EXPECTED_SHA256 = {
    "best_model_fold1.pt":  "d8fc080614798a363ad9c36fbf6dd9783e9b445073fe77e9e3fc58f9644eda32",
    "best_model_fold2.pt":  "07104cca50b27ae512f15728b8634c5edebec5bd07cd8a9d70d33f0addf28446",
    "best_model_fold3.pt":  "6779803fd8085cd725e842dbd3d37d5092e818ce4510effd2297d505e3dc4f71",
    "best_model_fold4.pt":  "a0ab4ebce5f321df63ff6628c0eb950e3c49672f9e3af177e6e70272bc72e657",
    "best_model_fold5.pt":  "083e0649067b29f79a9b060de19e3456d6b07e89ee41bdbe07ca5258463d8663",
    "best_model_global.pt": "20cb99afb0f7455d3c2a71494b586930439b761bbb23482d1d05a977955186d0",
}

HKA_KEY_COL   = "barcode"        # == DICOM filename stem
HKA_SIDE1_COL = "hka_side1"
HKA_SIDE2_COL = "hka_side2"

HKA_NEUTRAL_180    = False       # OAISYS convention: signed, 0 = neutral, negative = varus
IMAGE_LEFT_IS_SIDE = "1"
HKA_SIGN_FLIP      = False
DICOM_INVERT       = False
EXCLUDE_ABS_DEV    = 25.0        # |predicted HKA| beyond this (deg) => gross failure; flagged & excluded
NEUTRAL_BAND       = 3.0

OUT_DIR         = "oai_multimodel"   # all outputs go here (next to this notebook)
RERUN_INFERENCE = False              # True = ignore the cached predictions and run every model again
PUBLISHED_SINGLE_MODEL_CSV = "hto_oai_external_validation.csv"   # v2 single-model results, for the consistency check

os.makedirs(OUT_DIR, exist_ok=True)
WEIGHTS_DIR = next((d for d in WEIGHTS_DIR_CANDIDATES if os.path.isdir(d)), None)
assert WEIGHTS_DIR is not None, (
    "weights/ not found. Add '- ./weights:/tf/weights' under volumes: in docker-compose.yml "
    "and restart the container.")
_missing = [f for _, f, _ in MODELS if not os.path.isfile(os.path.join(WEIGHTS_DIR, f))]
assert not _missing, f"missing in {WEIGHTS_DIR}: {_missing}"
print(f"weights: {os.path.abspath(WEIGHTS_DIR)} ({len(MODELS)} models) | outputs: {os.path.abspath(OUT_DIR)}")

## Weight Manifest — SHA-256 of every model

Every metric below is tagged with the hash of the weights that produced it. Quote these hashes (or at least the global model's) in the paper's Code availability statement.

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

_rows = []
for label, fname, role in MODELS:
    p  = os.path.join(WEIGHTS_DIR, fname)
    st = os.stat(p)
    digest   = sha256_file(p)
    expected = EXPECTED_SHA256.get(fname)
    _rows.append(dict(model=label, role=role, file=fname, bytes=st.st_size,
                      modified=datetime.datetime.fromtimestamp(st.st_mtime).isoformat(timespec="seconds"),
                      sha256=digest,
                      matches_expected=(expected is None) or (digest == expected)))
MANIFEST = pd.DataFrame(_rows)
SHA = dict(zip(MANIFEST["model"], MANIFEST["sha256"]))

with pd.option_context("display.max_colwidth", 80, "display.width", 200):
    print(MANIFEST[["model", "file", "bytes", "modified", "sha256", "matches_expected"]].to_string(index=False))
MANIFEST.to_csv(os.path.join(OUT_DIR, "weights_manifest.csv"), index=False)
print(f"\nSaved: {OUT_DIR}/weights_manifest.csv")

if not MANIFEST["matches_expected"].all():
    bad = MANIFEST.loc[~MANIFEST["matches_expected"], "file"].tolist()
    print(f"\nWARNING: hash mismatch for {bad}. These are not the weights recorded in EXPECTED_SHA256; "
          "results will not correspond to the recorded models.")
else:
    print("All six files match the recorded SHA-256.")

## Image Loading & Preprocessing
Identical to the single-model notebook.

In [ ]:
def preprocess_global_image(img, target_size=512):
    """Letterbox-resize *img* to a square canvas of *target_size* pixels."""
    orig_w, orig_h = img.size
    scale   = min(target_size / orig_w, target_size / orig_h)
    new_w   = int(orig_w * scale)
    new_h   = int(orig_h * scale)
    resized = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    pad_left = (target_size - new_w) // 2
    pad_top  = (target_size - new_h) // 2
    final_img = Image.new("RGB", (target_size, target_size), (0, 0, 0))
    final_img.paste(resized, (pad_left, pad_top))
    return final_img, scale, (pad_left, pad_top)


def load_oai_image(path):
    """Load an OAI radiograph as an RGB PIL image (handles DICOM windowing / inversion)."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".dcm", ".dicom", ""):
        import pydicom
        ds  = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        slope = ds.get("RescaleSlope", 1.0);     slope = 1.0 if slope in (None, "") else float(slope)
        inter = ds.get("RescaleIntercept", 0.0); inter = 0.0 if inter in (None, "") else float(inter)
        arr = arr * slope + inter
        lo, hi = np.percentile(arr, (1, 99))                              # robust window
        arr = np.clip((arr - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
        if ds.get("PhotometricInterpretation", "") == "MONOCHROME1" or DICOM_INVERT:
            arr = 1.0 - arr
        return Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    return Image.open(path).convert("RGB")


_IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
def to_model_tensor(pil):
    t = torch.from_numpy(np.array(pil)).permute(2, 0, 1).float() / 255.0
    return (t - _IMNET_MEAN) / _IMNET_STD

## HKA from Predicted Landmarks
Laterality-consistent sign (fix from commit `da2c1a1`), with the straight-leg and mirror-leg self-tests.

In [ ]:
# Slots within each hemisphere: 0 femur_head | 1 knee_inner | 2 ost_point | 3 knee_outer
#                               4 ankle_inner | 5 ankle_outer   (ost_point unused for HKA)
def hka_from_side(pts6, neutral180=None, sign_flip=None):
    if neutral180 is None: neutral180 = HKA_NEUTRAL_180
    if sign_flip is None:  sign_flip  = HKA_SIGN_FLIP
    fh = pts6[0]; ki = pts6[1]; ko = pts6[3]
    kc = 0.5 * (ki + ko); ac = 0.5 * (pts6[4] + pts6[5])
    vf = fh - kc; vt = ac - kc                                  # knee->hip, knee->ankle
    cross = vf[0]*vt[1] - vf[1]*vt[0]; dot = vf[0]*vt[0] + vf[1]*vt[1]
    ang = abs(math.degrees(math.atan2(cross, dot)))            # ~180 for a straight leg
    dev = 180.0 - ang
    # knee_outer(lateral) - knee_inner(medial) is THIS leg's medial->lateral direction, so the
    # same clinical deformity gets the same sign on the left and right legs.
    lat  = 1.0 if (ko[0] - ki[0]) >= 0 else -1.0
    sign = (-1.0 if cross < 0 else 1.0) * lat * (-1.0 if sign_flip else 1.0)
    return (180.0 - sign*dev) if neutral180 else (sign*dev)


def predict_landmarks(model, x):
    """Preprocessed [1,3,768,768] tensor -> [12,2] landmark coords in 768 letterbox space."""
    with torch.no_grad():
        hms = torch.sigmoid(model(x))
    return extract_coordinates(hms.cpu(), scale_factor=1.0 / HEATMAP_SCALE)[0].numpy()


_straight = np.array([[100,0],[90,500],[0,0],[110,500],[90,1000],[110,1000]], float)
assert abs(hka_from_side(_straight, neutral180=True, sign_flip=False) - 180.0) < 0.1, "HKA self-test failed"
_vL = np.array([[400,100],[415,500],[0,0],[385,500],[450,900],[430,900]], float)  # image-left,  varus
_vR = np.array([[400,100],[385,500],[0,0],[415,500],[350,900],[370,900]], float)  # image-right, mirror varus
assert abs(hka_from_side(_vL) - hka_from_side(_vR)) < 1e-6, "left/right sign inconsistent"
print("HKA self-tests passed (straight -> 180; mirror legs share sign).")

## Load All Models

In [ ]:
_model_map = {
    "tiny":      Conformer_tiny_patch16_keypoint_half_heatmap,
    "small_p16": Conformer_small_patch16_keypoint_half_heatmap,
    "small_p32": Conformer_small_patch32_keypoint_half_heatmap,
    "base":      Conformer_base_patch16_keypoint_half_heatmap,
}

models = {}
for label, fname, role in MODELS:
    m = _model_map[MODEL_VARIANT](num_keypoints=12).to(device)
    m.load_state_dict(torch.load(os.path.join(WEIGHTS_DIR, fname), weights_only=True, map_location=device))
    m.eval()
    models[label] = m
    print(f"loaded {label:7s} ({role:6s}) {fname:22s} sha256 {SHA[label][:16]}…")

## Load OAI HKA Ground Truth

In [ ]:
assert os.path.exists(OAI_HKA_CSV), f"Ground-truth CSV not found at '{OAI_HKA_CSV}' — run prepare_oai_hka.py first."
hka_df     = pd.read_csv(OAI_HKA_CSV, dtype={HKA_KEY_COL: str})
hka_lookup = hka_df.set_index(HKA_KEY_COL)
nboth = int(hka_df[[HKA_SIDE1_COL, HKA_SIDE2_COL]].notna().all(axis=1).sum())
print(f"Loaded {len(hka_df)} full-limb images ({nboth} with both knees).")
oai_reader_sd = None
if {"readsd_side1", "readsd_side2"}.issubset(hka_df.columns):
    oai_reader_sd = pd.concat([hka_df["readsd_side1"], hka_df["readsd_side2"]]).dropna().mean()
    print(f"OAI within-knee reader agreement (SD): {oai_reader_sd:.3f} deg")

## Run Inference — one pass over the images, all models per image

Each radiograph is decoded and preprocessed once, then passed through every model. Per-knee predictions are cached in `oai_multimodel/per_knee_predictions.csv` together with each model's SHA-256. The cache is reused only if the hashes match the current weights, so a changed weight file always triggers a fresh run.

In [ ]:
IMG_EXTS = (".dcm", ".dicom", ".png", ".jpg", ".jpeg", ".tif", ".tiff")
NEUTRAL  = 180.0 if HKA_NEUTRAL_180 else 0.0
CACHE    = os.path.join(OUT_DIR, "per_knee_predictions.csv")

def _cache_matches(path):
    if not os.path.exists(path):
        return False
    c = pd.read_csv(path, usecols=["model", "sha256"])
    return dict(c.groupby("model")["sha256"].first()) == SHA

if not RERUN_INFERENCE and _cache_matches(CACHE):
    P = pd.read_csv(CACHE, dtype={"barcode": str, "side": str})
    print(f"Loaded cached predictions ({len(P)} rows); hashes match the current weights.")
else:
    paths = sorted(p for p in glob.glob(os.path.join(OAI_IMAGE_DIR, "*"))
                   if os.path.splitext(p)[1].lower() in IMG_EXTS)
    print(f"Found {len(paths)} images; running {len(models)} models per image...")
    try:
        from tqdm.auto import tqdm
        pbar = tqdm(paths, desc="OAI inference", unit="img")
    except Exception:
        pbar = paths

    rows, n_matched, n_load_failed = [], 0, 0
    infer_failed = {label: 0 for label in models}
    for path in pbar:
        stem = os.path.splitext(os.path.basename(path))[0]
        if stem not in hka_lookup.index:
            continue
        n_matched += 1
        row = hka_lookup.loc[stem]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        gt = {"1": pd.to_numeric(row.get(HKA_SIDE1_COL), errors="coerce"),
              "2": pd.to_numeric(row.get(HKA_SIDE2_COL), errors="coerce")}
        try:
            canon, _, _ = preprocess_global_image(load_oai_image(path), TARGET_SIZE)
            x = to_model_tensor(canon).unsqueeze(0).to(device)
        except Exception:
            n_load_failed += 1
            continue
        for label, model in models.items():
            try:
                coords = predict_landmarks(model, x)
            except Exception:
                infer_failed[label] += 1
                continue
            p_left, p_right = hka_from_side(coords[0:6]), hka_from_side(coords[6:12])
            pred_side = ({"1": p_left, "2": p_right} if IMAGE_LEFT_IS_SIDE == "1"
                         else {"1": p_right, "2": p_left})
            for side in ("1", "2"):
                if pd.isna(gt[side]):
                    continue
                pred = float(pred_side[side]); ref = float(gt[side])
                rows.append(dict(model=label, role=ROLE[label], sha256=SHA[label],
                                 barcode=stem, side=side, pred_hka=pred, oai_hka=ref,
                                 abs_err=abs(pred - ref),
                                 flagged=bool(abs(pred - NEUTRAL) > EXCLUDE_ABS_DEV)))
    P = pd.DataFrame(rows)
    P.to_csv(CACHE, index=False)
    print(f"Matched {n_matched} images | image load failures: {n_load_failed} | "
          f"inference failures per model: {infer_failed}")
    print(f"Saved: {CACHE}")

# one CSV per model, in the same format as the single-model notebook's output
for label in models:
    (P[P["model"] == label][["barcode", "side", "pred_hka", "oai_hka", "abs_err", "flagged"]]
       .to_csv(os.path.join(OUT_DIR, f"oai_external_validation_{label}.csv"), index=False))

print(P.groupby("model", sort=False).agg(knees=("barcode", "size"), flagged=("flagged", "sum")).T.to_string())

## Consistency Check — global model vs the published single-model run

The global model here should reproduce `hto_oai_external_validation.csv` (the v2 results in the paper). Agreement confirms that the published numbers came from exactly these weights; disagreement means they did not.

In [ ]:
if os.path.exists(PUBLISHED_SINGLE_MODEL_CSV):
    ref = pd.read_csv(PUBLISHED_SINGLE_MODEL_CSV, dtype={"barcode": str, "side": str})
    g   = P[P["model"] == "global"]
    mm  = ref.merge(g, on=["barcode", "side"], suffixes=("_published", ""))
    same = np.isclose(mm["pred_hka_published"], mm["pred_hka"], atol=1e-4)
    max_dev = float(np.abs(mm["pred_hka_published"] - mm["pred_hka"]).max()) if len(mm) else float("nan")
    print(f"global model vs published run: {same.sum()}/{len(mm)} knees identical "
          f"({same.mean():.1%}); max |difference| = {max_dev:.2e} deg")
    if same.all() and len(mm) == len(ref):
        print(f"=> the published OAI results were produced by best_model_global.pt (sha256 {SHA['global']}).")
    else:
        print("=> NOT identical: the published OAI results do not come from these global weights.")
else:
    print(f"'{PUBLISHED_SINGLE_MODEL_CSV}' not found next to this notebook — check skipped.")

## Per-Model Agreement

Each model is scored on its own non-flagged knees (the protocol used in the paper), and again on the **common set** — knees that no model flagged — so every model is compared on identical knees.

In [ ]:
def icc21(M):
    X = np.asarray(M, float); n, k = X.shape; g = X.mean()
    SSR = k * ((X.mean(1) - g) ** 2).sum(); SSC = n * ((X.mean(0) - g) ** 2).sum()
    SSE = ((X - g) ** 2).sum() - SSR - SSC
    MSR = SSR / (n - 1); MSC = SSC / (k - 1); MSE = SSE / ((n - 1) * (k - 1))
    return (MSR - MSE) / (MSR + (k - 1) * MSE + (k / n) * (MSC - MSE))

def agreement(pred, oai):
    diff = pred - oai
    bias = diff.mean(); sd = diff.std(ddof=1)
    ref  = 180.0 if HKA_NEUTRAL_180 else 0.0; nb = NEUTRAL_BAND
    strata = {"varus": oai < ref - nb, "neutral": np.abs(oai - ref) <= nb, "valgus": oai > ref + nb}
    out = dict(n=len(diff), mae=np.abs(diff).mean(), rmse=float(np.sqrt((diff ** 2).mean())),
               bias=bias, loa_lo=bias - 1.96 * sd, loa_hi=bias + 1.96 * sd, loa_half=1.96 * sd,
               icc=icc21(np.c_[pred, oai]), pearson_r=pearsonr(pred, oai)[0],
               spearman_rho=spearmanr(pred, oai)[0],
               magnitude_r=pearsonr(np.abs(pred), np.abs(oai))[0])
    for name, m in strata.items():
        out[f"mae_{name}"] = np.abs(diff[m]).mean() if m.any() else np.nan
        out[f"n_{name}"]   = int(m.sum())
    return out

KEY = ["barcode", "side"]
_fs = P.groupby(KEY)["flagged"].agg(["sum", "size"])
_unflagged_everywhere = _fs[(_fs["sum"] == 0) & (_fs["size"] == len(models))].reset_index()[KEY]
print(f"common set: {len(_unflagged_everywhere)} knees unflagged by all {len(models)} models")

_rows = []
for label, fname, role in MODELS:
    D = P[P["model"] == label]
    V = D[~D["flagged"]]
    a = agreement(V["pred_hka"].values, V["oai_hka"].values)
    C = D.merge(_unflagged_everywhere, on=KEY)
    c = agreement(C["pred_hka"].values, C["oai_hka"].values)
    _rows.append(dict(model=label, role=role, sha256=SHA[label], knees_total=len(D),
                      flagged=int(D["flagged"].sum()), flagged_pct=100 * D["flagged"].mean(),
                      **a, **{f"common_{k}": c[k] for k in ("n", "mae", "icc", "pearson_r", "bias", "loa_half")}))
PM = pd.DataFrame(_rows)
PM.to_csv(os.path.join(OUT_DIR, "per_model_metrics.csv"), index=False)

_show = PM[["model", "flagged", "flagged_pct", "n", "mae", "rmse", "bias", "loa_half", "icc",
            "pearson_r", "spearman_rho", "mae_varus", "mae_neutral", "mae_valgus",
            "common_mae", "common_icc"]]
print("\nOwn non-flagged knees (paper protocol) + common-set MAE / ICC")
print(_show.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
if oai_reader_sd is not None:
    print(f"\nreference: OAI inter-reader SD = {oai_reader_sd:.2f} deg")
print(f"\nSaved: {OUT_DIR}/per_model_metrics.csv")

## Aggregate Results

Fold models are five independently trained models (each on 80% of the training data), so their spread estimates run-to-run variability of the recipe. The global model is reported alongside and placed against that spread.

In [ ]:
METRICS = ["flagged_pct", "n", "mae", "rmse", "bias", "loa_half", "icc", "pearson_r", "spearman_rho",
           "magnitude_r", "mae_varus", "mae_neutral", "mae_valgus", "common_mae", "common_icc"]
F = PM[PM["role"] == "fold"].set_index("model")[METRICS]
G = PM[PM["role"] == "global"].set_index("model")[METRICS].iloc[0]
A = PM.set_index("model")[METRICS]

AGG = pd.DataFrame({
    "folds_mean": F.mean(), "folds_sd": F.std(ddof=1), "folds_min": F.min(), "folds_max": F.max(),
    "global": G,
    "all6_mean": A.mean(), "all6_sd": A.std(ddof=1),
})
AGG["global_within_fold_range"] = (G >= F.min()) & (G <= F.max())
AGG.index.name = "metric"
AGG.to_csv(os.path.join(OUT_DIR, "aggregate_metrics.csv"))
print(AGG.to_string(float_format=lambda v: f"{v:.3f}"))
print(f"\nSaved: {OUT_DIR}/aggregate_metrics.csv")

def _ms(k, d=2, unit=""):
    return f"{F[k].mean():.{d}f}{unit} ± {F[k].std(ddof=1):.{d}f}{unit} (range {F[k].min():.{d}f} to {F[k].max():.{d}f}{unit})"

print("\nSummary for the manuscript (fold models, mean ± SD, range):")
print(f"  ICC(2,1)        {_ms('icc', 3)}")
print(f"  Pearson r       {_ms('pearson_r', 3)}")
print(f"  MAE             {_ms('mae', 2, '°')}")
print(f"  bias            {_ms('bias', 2, '°')}")
print(f"  95% LoA (±)     {_ms('loa_half', 2, '°')}")
print(f"  gross failures  {_ms('flagged_pct', 1, '%')}")
print(f"  global model    ICC {G['icc']:.3f}, MAE {G['mae']:.2f}°, gross failures {G['flagged_pct']:.1f}% "
      f"(sha256 {SHA['global'][:12]}…)")

## Plots

In [ ]:
FOLD_C, GLOBAL_C = "#4C72B0", "#C44E52"

# (1) agreement scatter per model
fig, axes = plt.subplots(2, 3, figsize=(15, 9.5), sharex=True, sharey=True)
_lim = (-25, 25)
for ax, (label, fname, role) in zip(axes.ravel(), MODELS):
    V = P[(P["model"] == label) & (~P["flagged"])]
    row = PM.set_index("model").loc[label]
    ax.scatter(V["oai_hka"], V["pred_hka"], s=5, alpha=0.25,
               color=GLOBAL_C if role == "global" else FOLD_C, rasterized=True)
    ax.plot(_lim, _lim, "k--", lw=1, alpha=0.6)
    ax.set_xlim(_lim); ax.set_ylim(_lim); ax.grid(alpha=0.3)
    ax.set_title(f"{label}  ICC={row['icc']:.3f}  MAE={row['mae']:.2f}°  "
                 f"flagged={row['flagged_pct']:.1f}%\nsha256 {SHA[label][:12]}…", fontsize=9)
for ax in axes[1]: ax.set_xlabel("OAI clinical HKA (°)")
for ax in axes[:, 0]: ax.set_ylabel("Predicted HKA (°)")
fig.suptitle("OAI external validation — HKA agreement per model (non-flagged knees)")
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "agreement_per_model.png"), dpi=150); plt.show()

# (2) metric spread: folds as points with mean ± SD band, global highlighted
panels = [("icc", "ICC(2,1)"), ("mae", "MAE (°)"), ("loa_half", "95% LoA half-width (°)"), ("flagged_pct", "Gross failures (%)")]
fig, axes = plt.subplots(1, len(panels), figsize=(16, 3.6))
for ax, (k, title) in zip(axes, panels):
    fv = F[k].values; mu, sd = fv.mean(), fv.std(ddof=1)
    ax.axhspan(mu - sd, mu + sd, color=FOLD_C, alpha=0.12, lw=0)
    ax.axhline(mu, color=FOLD_C, lw=1)
    ax.scatter(range(len(fv)), fv, color=FOLD_C, zorder=3, label="fold models")
    ax.scatter([len(fv) + 0.5], [G[k]], color=GLOBAL_C, marker="D", s=50, zorder=3, label="global")
    ax.set_xticks(list(range(len(fv))) + [len(fv) + 0.5]); ax.set_xticklabels(list(F.index) + ["global"], rotation=45)
    ax.set_title(title); ax.grid(axis="y", alpha=0.3)
axes[0].legend(fontsize=8, loc="lower left")
fig.suptitle("Between-model spread (band = fold mean ± SD)")
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "metric_spread.png"), dpi=150); plt.show()
print(f"Saved: {OUT_DIR}/agreement_per_model.png, {OUT_DIR}/metric_spread.png")

## Between-Model Disagreement per Knee

For each knee on the common set, the SD of the predicted HKA across the five fold models. This is the per-knee size of the training-run variability, in degrees — compare it with the OAI inter-reader SD.

In [ ]:
FOLDS = [l for l, _, r in MODELS if r == "fold"]
W = (P.merge(_unflagged_everywhere, on=KEY)
       .pivot_table(index=KEY, columns="model", values="pred_hka"))
knee_sd = W[FOLDS].std(axis=1, ddof=1)
glob_dev = (W["global"] - W[FOLDS].mean(axis=1)).abs()

print(f"knees: {len(knee_sd)}")
print(f"between-fold SD of predicted HKA: median {knee_sd.median():.2f}°, "
      f"90th pct {knee_sd.quantile(0.9):.2f}°, >1°: {100*(knee_sd > 1).mean():.1f}%, >2°: {100*(knee_sd > 2).mean():.1f}%")
print(f"|global − fold mean|: median {glob_dev.median():.2f}°, 90th pct {glob_dev.quantile(0.9):.2f}°")
if oai_reader_sd is not None:
    print(f"reference: OAI inter-reader SD = {oai_reader_sd:.2f}°")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(knee_sd.clip(upper=5), bins=60, color=FOLD_C, alpha=0.8)
if oai_reader_sd is not None:
    ax.axvline(oai_reader_sd, color="k", ls="--", lw=1, label=f"OAI reader SD {oai_reader_sd:.2f}°"); ax.legend()
ax.set_xlabel("SD of predicted HKA across fold models (°, clipped at 5)"); ax.set_ylabel("knees")
ax.set_title("Per-knee between-model disagreement"); ax.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "between_model_sd.png"), dpi=150); plt.show()

W.assign(fold_sd=knee_sd, global_minus_fold_mean=W["global"] - W[FOLDS].mean(axis=1)) \
 .reset_index().to_csv(os.path.join(OUT_DIR, "per_knee_model_spread.csv"), index=False)
print(f"Saved: {OUT_DIR}/between_model_sd.png, {OUT_DIR}/per_knee_model_spread.csv")

## Outputs (`oai_multimodel/`)

| File | Contents |
| --- | --- |
| `weights_manifest.csv` | file, size, modified time and SHA-256 of each model |
| `per_knee_predictions.csv` | every knee × model: predicted and OAI HKA, error, flag, model hash (also the inference cache) |
| `oai_external_validation_<model>.csv` | per-model results in the single-model notebook's format |
| `per_model_metrics.csv` | all agreement metrics per model, own and common knee sets |
| `aggregate_metrics.csv` | fold mean / SD / range, global, all-six mean / SD |
| `per_knee_model_spread.csv` | per-knee predictions of all models with between-fold SD |
| `agreement_per_model.png`, `metric_spread.png`, `between_model_sd.png` | figures |